In [1]:
from platform import python_version
print(python_version())

3.11.14


### Cluster with Tahoe or sc-GTP



In [2]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"

sys.path.insert(0, ROOT_SRC)


if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import create_dir
from libs.MTD_lib import MTD
from libs.cBioPortal_lib import cBioPortal
from libs.calc_degs_lib import CALC_DEGS
# from libs.dashcyto_lib import DASH_CYTO
from libs.config_lib import Config
from libs.prism_lib import PRISM
from libs.prism_program_lib import *


from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/perturb_agent
ROOT_SRC added: /home/flavio/uv/perturb_agent/src


/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/Bio/__init__.py:138: BiopythonWarning: You may be importing Biopython from inside the source tree. This is bad practice and might lead to downstream issues. In particular, you might encounter ImportErrors due to missing compiled C extensions. We recommend that you try running your code from outside the source tree. If you are outside the source tree then you have a pyproject.toml file in an unexpected directory: /home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages
  warnings.warn(
/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

PROG_ID = 'TCGA'
PSI_ID = 'BRCA'
PSI_ID = 'ACC'
PSI_ID = 'CESC'
PSI_ID = 'PAAD'

ROOT0_DATA = ROOT0 / "data"
root_colab = ROOT0_DATA / 'colab'
root_project = ROOT0_DATA / PROG_ID

disease = PSI_ID

root_project = create_dir(ROOT0_DATA, s_project)
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']


case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=ROOT0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

Best parameter file for LFC does not exist /home/flavio/uv/perturb_agent/data/TCGA/PAAD/config/all_lfc_cutoffs_PAAD.tsv
project 'TCGA', s_project 'TCGA'
G/P LFC cutoffs: lfc=1.000; fdr=0.050 - LFC_cut_inf=0.400
Pathway cutoffs: pval=0.050; fdr=0.050; num of genes=3


In [4]:
mtd = MTD(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, 
          root0=ROOT0, root0_data=ROOT0_DATA, prog_id=PROG_ID, psi_id=PSI_ID,
          case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
          std_filename=std_filename, std_filename_list=std_filename_list,
          geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
          tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
          LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
          num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
          min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
          saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", mtd.root0, mtd.root_disease)
case = case_list[0]
print(">>>", mtd.disease, case)

mtd.cfg.set_default_best_lfc_cutoff(mtd.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, prompt_verbose=True, verbose=False)
print("\nEcho Parameters:")
print(mtd.echo_parameters())

>>> Roots /home/flavio/uv/perturb_agent /home/flavio/uv/perturb_agent/data/TCGA/PAAD
>>> PAAD Tumor
>>> case Tumor
>>> psi_id or disease: PAAD
Error: No data available for the specified PAAD.
Error: could not find /home/flavio/uv/perturb_agent/data/TCGA/PAAD/lfc/PAAD_final_LFC_Tumor_x_CTRL_not_normalized.tsv
No dflfc table was calculated for this case Tumor

Echo Parameters:
	0/0 DEGs/ensembl.
		Up 0/0 DEGs/ensembl.
		Dw 0/0 DEGs/ensembl.

Found 0 (best=3) pathways for geneset num=0 'Reactome_Pathways_2024'
Pathway cutoffs p-value=0.050 fdr=0.050 min genes=0.05No enrichment analysis was calculated.


In [5]:
cbio = cBioPortal(root0=ROOT0, root0_data=ROOT0_DATA, memory_restriction=False)

### Get all programs

In [6]:
verbose = False

df_psi = cbio.open_primary_site(verbose=verbose)
df_psi

,prog_id,cbioportal_study_id,active,gdc_project_id,psi_id,disease_id,disease_cd,primary_site,disease_context
0,TCGA,paad_tcga_pan_can_atlas_2018,True,TCGA-PAAD,PAAD,pancreatic_adenocarcinoma,PAAD,Pancreas,"TCGA pancreatic adenocarcinoma, PanCancer Atlas"
1,CPTAC3,paad_cptac_2021,True,CPTAC-3,PAAD,pancreatic_ductal_adenocarcinoma,PAAD,Pancreas,"CPTAC publication cohort, Cell 2021; 140 pancreatic cancers"
2,TCGA,skcm_tcga_pan_can_atlas_2018,True,TCGA-SKCM,SKCM,cutaneous_melanoma,SKCM,Skin,"TCGA skin cutaneous melanoma, PanCancer Atlas"
3,TCGA,brca_tcga_pan_can_atlas_2018,True,TCGA-BRCA,BRCA,breast_invasive_carcinoma,BRCA,Breast,"TCGA breast invasive carcinoma, PanCancer Atlas"
4,CPTAC2,brca_cptac_2020,True,CPTAC-2,BRCA,breast_cancer,BRCA,Breast,"CPTAC breast cancer publication cohort, Cell 2020"


### Open primary cites from cbio

In [7]:
PROG_ID = 'TCGA'
psi_id = 'PAAD'
psi_id = 'SKCM'
psi_id = 'BRCA'

PROG_ID = 'CPTAC2'
psi_id = 'BRCA'

PROG_ID = 'CPTAC3'
psi_id = 'PAAD'

### Prism - development

In [8]:
import anndata as ad

prism = PRISM(root0=ROOT0, root0_data=ROOT0_DATA)

verbose=True

prism.set_program_and_primary_site(prog_id=PROG_ID, psi_id=psi_id, verbose=verbose)

prism.root_singc, prism.root_singc.exists()

Table opened ((7, 9)) at '/home/flavio/uv/perturb_agent/data/cbioportal_study_mapping.tsv'

-----------------------------
>> prog_id: CPTAC3
>> psi_id: PAAD
>> primary_site: Pancreas
>> disease_id: pancreatic_ductal_adenocarcinoma
>> disease_cd: PAAD

-----------------------------
>> cbioportal_study_id: paad_cptac_2021
>> gdc_project_id: CPTAC-3

-----------------------------
>> root disease: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD
>> root samples: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/samples
>> root lfc: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/lfc
>> root mutations: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/mutations
-----------------------------



(PosixPath('/home/flavio/uv/perturb_agent/data/single_cell'), True)

### Running prism

In [9]:
verbose=True

res = prism.open_bayesprism(verbose=verbose)
print(len(res.genes))

Loaded /home/flavio/uv/perturb_agent/data/single_cell/deconv.h5ad (6.8 MB)
1604


In [10]:
res.states

['Fibroblast cell',
 'Stellate cell',
 'Macrophage cell',
 'Endothelial cell',
 'T cell',
 'B cell',
 'Ductal cell type 2',
 'Endocrine cell',
 'Ductal cell type 1',
 'Acinar cell']

In [11]:
res.theta_type.columns

Index(['Acinar cell', 'B cell', 'Ductal cell type 1', 'Endocrine cell', 'Endothelial cell',
       'Fibroblast cell', 'Macrophage cell', 'Stellate cell', 'T cell', 'malignant'],
      dtype='object')

### Ductal cell type 1

"Ductal cell type 1" is the normal-like ductal population and stays in the environment compartment — which is what you want. If both had been mapped to malignant, purity would inflate. Verify with res.tumor_purity.groupby(meta["condition"]).describe(): normals near zero, tumors somewhere in 0.2–0.6.


### Ductal cell type 2

One malignant state means no Ductal cell type 2 subdivision, so subtype_malignant scores Moffitt signatures on a single pooled malignant profile. That still works — it's per-sample expression, so samples can differ — but it won't give you distinct malignant states in θ. For that you'd subcluster Ductal cell type 2 in the AnnData and write finer cell_state labels before calling pseudobulk_reference.

In [12]:
res.cell_type_expression("Ductal cell type 1").shape

(1604, 153)

In [13]:
res.cell_type_expression("Ductal cell type 2").shape

(1604, 153)

### 2. theta is now fixed -> expand Z to every gene

In [14]:
verbose=False
force=False

imax_tumor=250
imax_normal=50

exclude_prog_list=['CCLE']
disease_cd = 'PAAD'

dfn_tumor, dfn_normal, df_gtex, df_summ = cbio.get_all_data_from_disease(disease_cd=disease_cd, 
                                                           imax_tumor=imax_tumor, imax_normal=imax_normal,
                                                           exclude_prog_list=exclude_prog_list,
                                                           force=force, verbose=verbose)

verbose=True
force=False

df_bulk, df_meta = prism.build_bulk_matrix(dfn_tumor, dfn_normal, cbio.df_metadata, 
                                        keep_biotypes=("protein_coding", "lncRNA", "miRNA"),
                                        force=force, verbose=verbose)

force=False
verbose=True
fname = "count-matrix.txt"
fname_ad = fname.replace('.txt', '.h5ad')

adata = prism.load_matrix(fname=fname, sep=' ', force=force, verbose=verbose)

filename_ad = prism.root_singc / fname_ad
compression = "gzip"
# adata.write_h5ad(filename_ad, compression=compression)
print(f"AData saved as {filename_ad},  ({filename_ad.stat().st_size/1e6:.0f} MB), compressed with {compression}")


verbose=True
fname_celltype = "all_celltype.txt"
adata = prism.attach_celltypes(adata=adata, fname_celltype=fname_celltype, verbose=verbose)

ref, s2t = prism.pseudobulk_reference(adata)

Error reading csv/tsv '/home/flavio/uv/perturb_agent/data/multi_progs/PAAD/lfc/expression_gtex_controls_counts.tsv': No columns to parse from file
Table opened ((27169, 153)) at '/home/flavio/uv/perturb_agent/data/single_cell/bulk_matrix.tsv'
Table opened ((153, 4)) at '/home/flavio/uv/perturb_agent/data/single_cell/bulk_metadata.tsv'
57,530 cells x 24,005 genes | obs: []
AData saved as /home/flavio/uv/perturb_agent/data/single_cell/count-matrix.h5ad,  (338 MB), compressed with gzip
all_celltype.txt columns: ['cluster']
                             cluster
cell.name                           
T1_AAACCTGAGATGTCGG  Fibroblast cell
T1_AAACGGGGTCATGCAT    Stellate cell
T1_AAAGATGCATGTTGAC  Macrophage cell
using type_col='cluster'
barcode overlap: 57,530 / 57,530
cell_type
malignant             11315
Ductal cell type 1    10317
Endothelial cell       9117
Fibroblast cell        6742
Stellate cell          5907
Macrophage cell        5361
T cell                 3660
B cell                 24

In [15]:
Zfull, gfull = prism.full_Z(res, df_bulk, ref)

In [16]:
dic = {}

for cell_state in res.states:
    print(cell_state)
    Z = prism.state_expression(Zfull, gfull, res, cell_state)
    dic[cell_state] = Z

Fibroblast cell
Stellate cell
Macrophage cell
Endothelial cell
T cell
B cell
Ductal cell type 2
Endocrine cell
Ductal cell type 1
Acinar cell


In [17]:
i=0
key = list(dic.keys())[i]

print(key)
dic[key]

Fibroblast cell


,T-C3L-02890,T-C3L-03635,T-C3L-02701,T-C3L-04072,T-C3L-00589,T-C3L-03123,T-C3N-01383,T-C3L-01124,T-C3L-00625,T-C3N-03439,...,N-C3N-03069,N-C3N-02765,N-C3L-07037,N-C3N-02589,N-C3N-02996,N-C3L-02606,N-C3N-03173,N-C3N-02696,N-TCGA-H6-8124,N-TCGA-H6-A45N
A1BG,6.551e-01,4.418e-01,6.680e-01,2.693e-01,4.914e-01,4.378e-01,1.043e+00,8.103e-01,9.495e-01,7.379e-01,...,NaN,0.772,0.328,NaN,NaN,NaN,1.515e-01,NaN,2.648e-01,NaN
A1BG-AS1,3.150e+00,1.251e+00,1.249e+00,2.249e+00,1.408e+00,1.848e+00,3.725e+00,1.988e+00,2.017e+00,2.263e+00,...,NaN,6.944,9.022,NaN,NaN,NaN,7.544e+00,NaN,1.126e+00,NaN
A1CF,3.463e+00,1.384e+00,2.188e+00,1.346e-01,1.305e+00,8.334e-02,1.176e+00,2.169e+00,1.218e+00,3.142e+00,...,NaN,1.033,3.148,NaN,NaN,NaN,1.032e+01,NaN,1.701e+00,NaN
A2M,1.344e+03,1.258e+03,1.165e+03,8.277e+02,1.477e+03,8.272e+02,1.130e+03,1.373e+03,2.035e+03,2.286e+03,...,NaN,1083.582,1060.576,NaN,NaN,NaN,2.068e+03,NaN,2.003e+03,NaN
A2M-AS1,6.254e+00,7.056e+00,2.835e+00,2.690e+00,5.443e+00,7.466e+00,6.568e+00,7.522e+00,6.423e+00,1.268e+01,...,NaN,2.626,7.625,NaN,NaN,NaN,1.319e+01,NaN,2.975e+00,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZYG11A,8.548e-09,2.341e-08,6.967e-08,7.866e-09,2.045e-08,2.942e-08,6.440e-08,2.001e-08,1.809e-07,1.226e-08,...,NaN,0.021,0.000,NaN,NaN,NaN,7.822e-08,NaN,6.635e-09,NaN
ZYG11B,1.060e+02,1.070e+02,1.151e+02,8.354e+01,1.278e+02,9.075e+01,1.159e+02,1.173e+02,9.685e+01,1.466e+02,...,NaN,53.810,83.431,NaN,NaN,NaN,1.003e+02,NaN,5.187e+01,NaN
ZYX,7.451e+01,5.875e+01,7.764e+01,1.232e+02,4.028e+01,7.744e+01,5.810e+01,6.771e+01,7.150e+01,5.684e+01,...,NaN,87.890,61.471,NaN,NaN,NaN,3.394e+01,NaN,3.417e+02,NaN
ZZEF1,1.460e+02,1.129e+02,9.541e+01,1.231e+02,1.044e+02,1.199e+02,1.055e+02,1.418e+02,1.251e+02,1.251e+02,...,NaN,129.316,99.746,NaN,NaN,NaN,5.434e+02,NaN,8.607e+01,NaN


### Ductal 2 - malignant

In [18]:
Zmal = prism.state_expression(Zfull, gfull, res, "Ductal cell type 2")

In [19]:
for g in ["FAM83A-AS1", "HOXA10-AS", "HOXB-AS3", "MIR7-3HG"]:
    if g in gfull:
        print(g, prism.gene_compartment_share(Zfull, gfull, res, g).head(3).round(3).to_dict())

FAM83A-AS1 {'Fibroblast cell': nan, 'Stellate cell': nan, 'Macrophage cell': nan}
HOXA10-AS {'Fibroblast cell': nan, 'Stellate cell': nan, 'Macrophage cell': nan}
HOXB-AS3 {'Fibroblast cell': nan, 'Stellate cell': nan, 'Macrophage cell': nan}
MIR7-3HG {'Fibroblast cell': nan, 'Stellate cell': nan, 'Macrophage cell': nan}


In [20]:
prog1 = ["FAM83A-AS1", "HOXA10-AS", "HOXB-AS3", "HOXB-AS4", "MIR7-3HG"]

prog2 = ["GATA6", "KRT17", "NEAT1", "H19", "DLEU1", "DLEU2"]

### survived build_bulk_matrix?

> Almost certainly df_bulk is the culprit: build_bulk_matrix defaults to keep_biotypes=("protein_coding",), which removes every lncRNA. Rebuild with them included:

In [21]:
[g for g in prog1 if g in df_bulk.index]

['FAM83A-AS1', 'HOXA10-AS', 'HOXB-AS3', 'HOXB-AS4', 'MIR7-3HG']

### present in the scRNA reference?

In [22]:
  
[g for g in prog1 if g in ref.columns]

['FAM83A-AS1', 'HOXA10-AS', 'HOXB-AS3', 'MIR7-3HG']

### Data treatment

1. get raw dfc
2. filter low-expression genes
3. normalize for library size
4. variance-stabilizing transformation
5. select most variable genes
6. cluster samples into k = 3..8 groups
7. evaluate clusters
8. find gene signatures for each cluster

A low-expression gene can be biologically important and even differentially expressed, especially if it is a transcription factor, cytokine, receptor, lncRNA, or rare-cell marker.

But for unsupervised tumor clustering, we usually do not want thousands of genes with mostly zero/very low counts because they add noise and unstable distances.

In [23]:
set(ref.index.to_list())

{'Acinar cell',
 'B cell',
 'Ductal cell type 1',
 'Ductal cell type 2',
 'Endocrine cell',
 'Endothelial cell',
 'Fibroblast cell',
 'Macrophage cell',
 'Stellate cell',
 'T cell'}

In [24]:
set(s2t.index.to_list())

{'Acinar cell',
 'B cell',
 'Ductal cell type 1',
 'Ductal cell type 2',
 'Endocrine cell',
 'Endothelial cell',
 'Fibroblast cell',
 'Macrophage cell',
 'Stellate cell',
 'T cell'}

In [25]:
adata.obs

,cluster,cell_type,cell_state
cell,,,
T1_AAACCTGAGATGTCGG,Fibroblast cell,Fibroblast cell,Fibroblast cell
T1_AAACGGGGTCATGCAT,Stellate cell,Stellate cell,Stellate cell
T1_AAAGATGCATGTTGAC,Macrophage cell,Macrophage cell,Macrophage cell
T1_AAAGATGGTCGAGTTT,Macrophage cell,Macrophage cell,Macrophage cell
T1_AAAGATGGTCTCTCTG,Endothelial cell,Endothelial cell,Endothelial cell
...,...,...,...
N11_TTTGCGCGTGCGCTTG,Endothelial cell,Endothelial cell,Endothelial cell
N11_TTTGGTTCATTGAGCT,Acinar cell,Acinar cell,Acinar cell
N11_TTTGGTTGTCCGACGT,Ductal cell type 1,Ductal cell type 1,Ductal cell type 1


In [26]:
import re, numpy as np, pandas as pd

adata.obs["sample"] = adata.obs_names.to_series().str.extract(r"^([TN]\d+)_")[0].values
adata.obs["tissue"] = np.where(adata.obs["sample"].str.startswith("T"), "tumor", "normal")

print(adata.obs.groupby("tissue")["sample"].nunique())      # expect tumor 24, normal 11
print(pd.crosstab(adata.obs["cell_state"], adata.obs["tissue"]))

tissue
normal    11
tumor     24
Name: sample, dtype: int64
tissue              normal  tumor
cell_state                       
Acinar cell           1423    512
B cell                  31   2416
Ductal cell type 1    7671   2646
Ductal cell type 2       0  11315
Endocrine cell         270    459
Endothelial cell      3983   5134
Fibroblast cell        940   5802
Macrophage cell        559   4802
Stellate cell          623   5284
T cell                  44   3616


### Count Malignant Cells - accordingo to transcriptomics

In [27]:
d2   = adata.obs["cell_state"].eq("Ductal cell type 2")
print(len(d2))
d2[:5]

57530


cell
T1_AAACCTGAGATGTCGG    False
T1_AAACGGGGTCATGCAT    False
T1_AAAGATGCATGTTGAC    False
T1_AAAGATGGTCGAGTTT    False
T1_AAAGATGGTCTCTCTG    False
Name: cell_state, dtype: bool

In [28]:
is_t = adata.obs["tissue"].eq("tumor")
print(np.sum(is_t))

41986


In [29]:
adata.obs["cell_state"] = np.where(d2 &  is_t, "Malignant ductal",
                          np.where(d2 & ~is_t, "Ductal cell type 2 normal",
                                   adata.obs["cell_state"]))
adata.obs

,cluster,cell_type,cell_state,sample,tissue
cell,,,,,
T1_AAACCTGAGATGTCGG,Fibroblast cell,Fibroblast cell,Fibroblast cell,T1,tumor
T1_AAACGGGGTCATGCAT,Stellate cell,Stellate cell,Stellate cell,T1,tumor
T1_AAAGATGCATGTTGAC,Macrophage cell,Macrophage cell,Macrophage cell,T1,tumor
T1_AAAGATGGTCGAGTTT,Macrophage cell,Macrophage cell,Macrophage cell,T1,tumor
T1_AAAGATGGTCTCTCTG,Endothelial cell,Endothelial cell,Endothelial cell,T1,tumor
...,...,...,...,...,...
N11_TTTGCGCGTGCGCTTG,Endothelial cell,Endothelial cell,Endothelial cell,N11,normal
N11_TTTGGTTCATTGAGCT,Acinar cell,Acinar cell,Acinar cell,N11,normal
N11_TTTGGTTGTCCGACGT,Ductal cell type 1,Ductal cell type 1,Ductal cell type 1,N11,normal


In [30]:
from collections import Counter

Counter(adata.obs["cell_state"] )

Counter({'Malignant ductal': 11315,
         'Ductal cell type 1': 10317,
         'Endothelial cell': 9117,
         'Fibroblast cell': 6742,
         'Stellate cell': 5907,
         'Macrophage cell': 5361,
         'T cell': 3660,
         'B cell': 2447,
         'Acinar cell': 1935,
         'Endocrine cell': 729})

In [31]:
adata.obs["cell_type"]  = np.where(adata.obs["cell_state"].eq("Malignant ductal"),
                                   "malignant", adata.obs["cell_type"])

Counter(adata.obs["cell_type"] )

Counter({'malignant': 11315,
         'Ductal cell type 1': 10317,
         'Endothelial cell': 9117,
         'Fibroblast cell': 6742,
         'Stellate cell': 5907,
         'Macrophage cell': 5361,
         'T cell': 3660,
         'B cell': 2447,
         'Acinar cell': 1935,
         'Endocrine cell': 729})

In [32]:
ref2, s2t = prism.pseudobulk_reference(adata, state_key="cell_state", type_key="cell_type")
s2t.to_dict()

{'Fibroblast cell': 'Fibroblast cell',
 'Stellate cell': 'Stellate cell',
 'Macrophage cell': 'Macrophage cell',
 'Endothelial cell': 'Endothelial cell',
 'T cell': 'T cell',
 'B cell': 'B cell',
 'Malignant ductal': 'malignant',
 'Endocrine cell': 'Endocrine cell',
 'Ductal cell type 1': 'Ductal cell type 1',
 'Acinar cell': 'Acinar cell'}

### LFC

calc_celltype_lfc() — each compartment vs the mean of the others, paired across samples by default. Paired is the right default here because every sample contributes every cell type, so pairing removes cohort/purity variance. This doubles as deconvolution QC: if the ductal compartment doesn't recover KRT19/TFF1/CEACAM6 and fibroblast doesn't recover COL1A1/POSTN, θ or the Peng reference is off and step 2 is meaningless.

### Critics

- Why not, for each cell type, tumor samples x normal samples
- Only Ductal 2 Tumor has no normal samples - to confirm


In [33]:
res.__dict__.keys()

dict_keys(['theta', 'theta_stage1', 'theta_type', 'tumor_purity', 'genes', 'Z', 'states'])

In [34]:
res.states

['Fibroblast cell',
 'Stellate cell',
 'Macrophage cell',
 'Endothelial cell',
 'T cell',
 'B cell',
 'Ductal cell type 2',
 'Endocrine cell',
 'Ductal cell type 1',
 'Acinar cell']

### Prism programs

In [35]:
Z_full, genes_full = prism.full_Z(res, df_bulk, ref)

### MalignantCluster

In [36]:
# del(MalignantCluster)

In [37]:
from libs.prism_malig_lib import MalignantCluster

In [38]:
type(res)

libs.prism_lib.DeconvResult

In [ ]:
root_mprog_cluster = create_dir(cbio.root_mprog_disease, 'cluster')

cell_name = "Ductal cell type 2"
kmax = 8
no_decouple = True
is_tahoe = True

mc = MalignantCluster(prism=prism, res=res, df_bulk=df_bulk, ref=ref, 
                      root_mprog_cluster=root_mprog_cluster, 
                      organ="Pancreas", cell_name=cell_name, cell_types=None)

In [40]:
d = mc.diagnose_filters()
print(d["Z_looks_like_counts"], d["Z_median_of_medians"])
print(d["by_min_counts"]); print(d["by_min_share"]); print(d["joint_grid"])

True 17.24835968017578
min_counts
0.0     16550
1.0     13266
5.0     10945
10.0     9666
50.0     4525
Name: n_genes, dtype: int64
min_share
0.0    16467
0.2    12999
0.3    10748
0.4     6538
0.5     3585
0.6     2376
0.7     1669
0.8     1066
Name: n_genes, dtype: int64
min_share     0.0    0.2    0.3   0.4   0.5   0.6   0.7   0.8
min_counts                                                   
0.0         16467  12999  10748  6538  3585  2376  1669  1066
1.0         13266  11505   9614  5661  2909  1838  1234   736
5.0         10945   9997   8457  4825  2326  1412   915   531
10.0         9666   9025   7743  4376  2040  1222   798   459
50.0         4525   4390   3983  2411  1115   660   434   262


### Huggingface: tahoebio/Tahoe-x1-embeddings

https://github.com/tahoebio/tahoe-x1

Tahoe-x1: Scaling Perturbation-Trained Single-Cell Foundation Models to 3 Billion Parameters


#### Memory

That's not a general "64 GB isn't enough" — swap is fully exhausted at 2.0G, which means something asked for tens of GB in one allocation. Given where you are in the pipeline, the culprit is almost certainly load_tahoe_de, and the arithmetic says so:

The DE table is ~4.09e9 rows over ~75k conditions × ~54k genes. 

Filtering to pancreas doesn't help much — roughly 
- 6 lines × 379 drugs × ~4 doses × 54k genes ≈ 5e8 rows, 
- materialised in pandas with gene/drug/cell_line_id as object-dtype strings (~200 B/row) before pivot_table ever runs. 
- That's >100 GB. full_Z and consensus_cluster are megabytes by comparison.


In [ ]:
X, diag = mc.prepare_malignant_matrix(keep_genes=mc.program1_panel)
print(X.shape)
X.head(3)

,A1CF,AACS,AADAC,AATK,ABAT,ABCA12,ABCA7,ABCB9,ABCC3,ABCC6,...,ZNF774,ZNF787,ZNF792,ZNF816,ZNF888,ZNRF1,ZNRF2,ZSCAN29,ZSWIM5,ZWINT
T-C3L-02890,6.339,6.394,5.173,5.321,5.810,3.304,6.691,5.138,7.894,3.771,...,3.659,5.141,4.990,4.945,6.294,4.822,5.624,5.873,4.213,5.154
T-C3L-03635,4.851,5.797,1.786,4.322,5.815,5.588,4.863,3.483,9.473,2.844,...,4.437,4.405,5.216,6.645,6.889,4.623,6.058,5.953,3.381,5.075
T-C3L-02701,5.459,6.252,2.637,5.297,5.755,4.061,6.086,3.164,8.562,3.482,...,4.564,4.759,4.606,5.259,6.386,4.851,5.279,5.854,4.099,5.349
T-C3L-04072,1.658,6.434,1.213,4.504,5.307,7.130,5.415,4.960,9.703,2.796,...,4.238,4.910,4.592,4.732,5.808,5.774,5.362,6.309,2.093,6.041
T-C3L-00589,4.531,6.285,5.176,5.432,5.450,6.509,6.044,4.678,9.594,4.098,...,4.179,4.901,5.093,4.609,5.911,4.628,5.299,6.107,3.360,4.386
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
N-C3N-01719,6.705,6.233,2.544,4.804,5.878,3.864,6.404,4.097,7.960,3.538,...,3.451,4.299,4.925,4.901,5.208,5.041,5.681,5.609,4.744,4.760
N-C3L-01689,7.512,5.832,6.638,7.046,7.113,2.299,5.794,4.209,9.582,7.092,...,4.332,4.702,4.964,4.471,5.238,5.046,5.109,6.278,5.112,4.121
N-C3N-01899,7.869,6.673,2.264,6.271,6.419,1.605,6.894,4.521,8.905,4.181,...,4.138,4.970,5.345,5.378,6.470,5.409,5.310,5.932,5.310,4.067
N-C3N-03173,7.676,5.147,7.042,5.576,7.205,2.294,4.595,3.446,9.114,6.194,...,3.533,3.787,5.389,4.913,5.503,4.418,5.745,6.396,4.591,4.480


In [ ]:
S, cond = mc.load_tahoe_de(organs=("Pancreas",), genes=X.columns)

In [ ]:
S

In [ ]:
out = mc.run(ks=range(2, kmax + 1),
            decouple_purity=not no_decouple,
            tahoe=is_tahoe)

print(f"selected k = {out['k']}")

df = pd.DataFrame({
    k: {"pac": v["pac"], "cophenetic": v["cophenetic"],
        "silhouette": v["silhouette"]}
    for k, v in out["consensus"].items()
}).T


In [ ]:
mc.download_tahoe(de=True)